# Component 5: The 1D U-Net Denoiser

The U-Net is the **core neural network** inside a diffusion policy. Its job is simple but critical: given a noisy action trajectory, predict the noise so we can remove it.

## Why a U-Net?

A U-Net is an **encoder-decoder** architecture with **skip connections**:

- **Encoder (downsampling path):** Compresses the input, capturing high-level patterns. Each block reduces the temporal resolution while increasing channels.
- **Decoder (upsampling path):** Reconstructs the output at the original resolution. Each block increases temporal resolution while decreasing channels.
- **Skip connections:** Connect encoder layers directly to decoder layers at the same resolution. This preserves fine-grained details that would otherwise be lost during compression.

## Why 1D Convolutions?

In image diffusion (like Stable Diffusion), the U-Net uses **2D convolutions** because the data is spatial (height x width). In diffusion policy, our data is a **sequence of actions over time** — shape `[batch, horizon, action_dim]`. We only need to convolve along the **time axis**, so we use **1D convolutions**.

Think of it like this:
- Image diffusion: denoise a 2D grid of pixels
- Action diffusion: denoise a 1D sequence of robot commands

In [ ]:
!pip install -q torch torchvision matplotlib numpy 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Build the 1D U-Net denoiser from scratch
# (same architecture pattern as used in the diffusion policy)
# ============================================================

class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for diffusion timesteps."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=x.device, dtype=torch.float32) * -emb)
        emb = x.unsqueeze(-1).float() * emb.unsqueeze(0)
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class ConditionalResidualBlock1D(nn.Module):
    """Residual block with FiLM conditioning for 1D sequences.
    
    Applies two conv1d layers with a residual skip connection.
    Conditioning (timestep + obs embedding) is injected via FiLM:
        h' = gamma * h + beta
    """
    def __init__(self, in_channels, out_channels, cond_dim):
        super().__init__()
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.GroupNorm(8, out_channels),
                nn.Mish(),
            ),
            nn.Sequential(
                nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.GroupNorm(8, out_channels),
                nn.Mish(),
            ),
        ])
        # FiLM: conditioning -> gamma, beta
        self.cond_encoder = nn.Sequential(
            nn.Mish(),
            nn.Linear(cond_dim, out_channels * 2),
        )
        # Residual connection (match channels if needed)
        self.residual_conv = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, cond):
        out = self.blocks[0](x)
        # FiLM conditioning
        cond_params = self.cond_encoder(cond)  # (B, 2*C)
        gamma, beta = cond_params.chunk(2, dim=-1)  # Each: (B, C)
        gamma = gamma.unsqueeze(-1)  # (B, C, 1) for broadcasting over time
        beta = beta.unsqueeze(-1)
        out = gamma * out + beta
        out = self.blocks[1](out)
        return out + self.residual_conv(x)


class ConditionalUnet1D(nn.Module):
    """1D U-Net for action diffusion.
    
    Architecture:
        - Encoder: Conv blocks that downsample temporal axis
        - Bottleneck: Conv blocks at lowest resolution
        - Decoder: Conv blocks that upsample back + skip connections
        - Conditioning injected via FiLM at every ResBlock
    
    Args:
        input_dim: action dimension (e.g., 2 for PushT x,y)
        global_cond_dim: dimension of the global conditioning vector
        down_dims: channel sizes for each encoder level
        diffusion_step_embed_dim: dimension of the timestep embedding
    """
    def __init__(self, input_dim=2, global_cond_dim=260, 
                 down_dims=(256, 512, 1024), diffusion_step_embed_dim=128):
        super().__init__()
        
        # Timestep encoder: scalar t -> embedding
        self.diffusion_step_encoder = nn.Sequential(
            SinusoidalPosEmb(diffusion_step_embed_dim),
            nn.Linear(diffusion_step_embed_dim, diffusion_step_embed_dim * 4),
            nn.Mish(),
            nn.Linear(diffusion_step_embed_dim * 4, diffusion_step_embed_dim),
        )
        
        # Total conditioning dim = timestep_emb + global_cond
        cond_dim = diffusion_step_embed_dim + global_cond_dim
        
        # Build encoder (down) path
        all_dims = [input_dim] + list(down_dims)
        self.down_modules = nn.ModuleList()
        for i in range(len(down_dims)):
            in_ch = all_dims[i]
            out_ch = all_dims[i + 1]
            self.down_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(in_ch, out_ch, cond_dim),
                ConditionalResidualBlock1D(out_ch, out_ch, cond_dim),
                nn.Conv1d(out_ch, out_ch, 3, stride=2, padding=1) if i < len(down_dims) - 1 else nn.Identity(),
            ]))
        
        # Bottleneck (mid) blocks
        mid_dim = down_dims[-1]
        self.mid_modules = nn.ModuleList([
            ConditionalResidualBlock1D(mid_dim, mid_dim, cond_dim),
            ConditionalResidualBlock1D(mid_dim, mid_dim, cond_dim),
        ])
        
        # Build decoder (up) path -- reverse of encoder
        self.up_modules = nn.ModuleList()
        for i in range(len(down_dims) - 1, -1, -1):
            out_ch = all_dims[i] if i > 0 else down_dims[0]
            in_ch = down_dims[i]
            skip_ch = down_dims[i]  # skip connection channels
            self.up_modules.append(nn.ModuleList([
                ConditionalResidualBlock1D(in_ch + skip_ch, out_ch, cond_dim),  # concat skip
                ConditionalResidualBlock1D(out_ch, out_ch, cond_dim),
                nn.ConvTranspose1d(out_ch, out_ch, 4, stride=2, padding=1) if i > 0 else nn.Identity(),
            ]))
        
        # Final projection back to action_dim
        self.final_conv = nn.Sequential(
            nn.Conv1d(down_dims[0], down_dims[0], 3, padding=1),
            nn.GroupNorm(8, down_dims[0]),
            nn.Mish(),
            nn.Conv1d(down_dims[0], input_dim, 1),
        )
    
    def forward(self, sample, timestep, local_cond=None, global_cond=None):
        """
        Args:
            sample: noisy actions [B, action_dim, horizon]
            timestep: diffusion timestep [B]
            global_cond: conditioning vector [B, cond_dim]
        Returns:
            predicted noise [B, action_dim, horizon]
        """
        # Encode timestep
        timestep_emb = self.diffusion_step_encoder(timestep)  # (B, embed_dim)
        
        # Combine with global conditioning
        if global_cond is not None:
            cond = torch.cat([timestep_emb, global_cond], dim=-1)
        else:
            cond = timestep_emb
        
        x = sample
        # Encoder: store skip connections
        skips = []
        for res1, res2, downsample in self.down_modules:
            x = res1(x, cond)
            x = res2(x, cond)
            skips.append(x)
            x = downsample(x)
        
        # Bottleneck
        for mid_block in self.mid_modules:
            x = mid_block(x, cond)
        
        # Decoder: use skip connections
        for res1, res2, upsample in self.up_modules:
            skip = skips.pop()
            # Handle size mismatch from downsampling
            if x.shape[-1] != skip.shape[-1]:
                x = F.interpolate(x, size=skip.shape[-1], mode='nearest')
            x = torch.cat([x, skip], dim=1)  # concat along channel dim
            x = res1(x, cond)
            x = res2(x, cond)
            x = upsample(x)
        
        return self.final_conv(x)


# ============================================================
# Instantiate with PushT-like configuration
# ============================================================
action_dim = 2          # PushT: (x, y)
horizon = 16            # 16 future actions
global_cond_dim = 260   # 128 (visual) + 4 (state) + 128 (timestep) = 260

unet = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=global_cond_dim,
    down_dims=(256, 512, 1024),
    diffusion_step_embed_dim=128,
).to(device)
unet.eval()

total_params = sum(p.numel() for p in unet.parameters())
print(f"1D U-Net built from scratch!")
print(f"Total parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB (float32)")
print(f"\nInput:  noisy actions [B, {action_dim}, {horizon}]")
print(f"Output: predicted noise [B, {action_dim}, {horizon}]")
print(f"Conditioning: {global_cond_dim}d global vector (visual + state + timestep)")

## Extracting the U-Net

The U-Net lives inside `policy.diffusion.unet`. Let's pull it out and inspect its architecture.

The key components are:
- **`down_modules`**: Encoder blocks that downsample the temporal axis
- **`mid_modules`**: Bottleneck blocks at the lowest resolution
- **`up_modules`**: Decoder blocks that upsample back to original resolution
- **`final_conv`**: A 1x1 convolution that maps back to the action dimension

In [ ]:
# Inspect the U-Net architecture

print("=" * 70)
print("U-Net Architecture Overview")
print("=" * 70)

# Count parameters
total_params = sum(p.numel() for p in unet.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB (float32)")

print("\n" + "=" * 70)
print("DOWN MODULES (Encoder)")
print("=" * 70)
for i, module in enumerate(unet.down_modules):
    params = sum(p.numel() for p in module.parameters())
    print(f"\n--- Down Block {i} ({params:,} params) ---")
    print(module)

print("\n" + "=" * 70)
print("MID MODULES (Bottleneck)")
print("=" * 70)
for i, module in enumerate(unet.mid_modules):
    params = sum(p.numel() for p in module.parameters())
    print(f"\n--- Mid Block {i} ({params:,} params) ---")
    print(module)

print("\n" + "=" * 70)
print("UP MODULES (Decoder)")
print("=" * 70)
for i, module in enumerate(unet.up_modules):
    params = sum(p.numel() for p in module.parameters())
    print(f"\n--- Up Block {i} ({params:,} params) ---")
    print(module)

print("\n" + "=" * 70)
print("FINAL CONV")
print("=" * 70)
print(unet.final_conv)

In [ ]:
# ============================================================
# Trace tensor shapes through the U-Net manually
# ============================================================
# The U-Net expects input in channel-first format: [B, C, T]
# where C = action_dim (2 for PushT) and T = horizon (16 steps)

print("=" * 70)
print("SHAPE TRACE THROUGH THE U-NET")
print("=" * 70)

# Create dummy inputs
batch_size = 1

# Noisy actions: [B, horizon, action_dim] -> rearrange to [B, action_dim, horizon]
noisy_actions = torch.randn(batch_size, horizon, action_dim).to(device)
print(f"\nInput noisy actions: {list(noisy_actions.shape)}  (batch, horizon, action_dim)")

# Rearrange for 1D conv: channels first
x = noisy_actions.permute(0, 2, 1)  # [B, action_dim, horizon]
print(f"After rearrange:     {list(x.shape)}  (batch, channels=action_dim, time=horizon)")

# We need a timestep embedding and global conditioning
timestep = torch.tensor([500]).to(device)  # Diffusion timestep

# Get the diffusion timestep embedding
diffusion_step_encoder = unet.diffusion_step_encoder
timestep_emb = diffusion_step_encoder(timestep)
print(f"\nTimestep embedding:  {list(timestep_emb.shape)}  (batch, embedding_dim)")

# Synthetic global conditioning (simulates obs encoder output)
# In the real policy: visual_features(128d) + state(4d) = 132d obs encoding
# But global_cond_dim = 260, so obs_cond would be 260d
obs_cond = torch.randn(batch_size, global_cond_dim).to(device)
print(f"Obs conditioning:    {list(obs_cond.shape)}  (batch, global_cond_dim)")

# Combined feature used internally
global_feature = torch.cat([timestep_emb, obs_cond], dim=-1)
print(f"Combined feature:    {list(global_feature.shape)}  (batch, timestep_dim + cond_dim)")

print("\n" + "-" * 70)
print("ENCODER (Down Path)")
print("-" * 70)

# Register hooks to capture shapes at key layers
hook_data = {}

def make_hook(name):
    def hook_fn(module, input, output):
        if isinstance(output, torch.Tensor):
            hook_data[name] = output.shape
        elif isinstance(output, tuple):
            hook_data[name] = [o.shape if isinstance(o, torch.Tensor) else type(o) for o in output]
    return hook_fn

hooks = []
for i, module in enumerate(unet.down_modules):
    hooks.append(module.register_forward_hook(make_hook(f'down_{i}')))
for i, module in enumerate(unet.mid_modules):
    hooks.append(module.register_forward_hook(make_hook(f'mid_{i}')))
for i, module in enumerate(unet.up_modules):
    hooks.append(module.register_forward_hook(make_hook(f'up_{i}')))
hooks.append(unet.final_conv.register_forward_hook(make_hook('final_conv')))

# Run a forward pass to capture shapes
with torch.no_grad():
    noisy_sample = torch.randn(1, action_dim, horizon).to(device)
    output = unet(noisy_sample, timestep, local_cond=None, global_cond=obs_cond)

print(f"\nObservation encoding: {list(obs_cond.shape)}  (synthetic)")
print(f"U-Net output: {list(output.shape)}  (batch, action_dim, horizon)")

# Print captured shapes
print("\nCaptured layer shapes:")
for name, shape in hook_data.items():
    print(f"  {name}: {shape}")

# After rearranging back
output_actions = output.permute(0, 2, 1)  # [B, horizon, action_dim]
print(f"\nFinal output (rearranged): {list(output_actions.shape)}  (batch, horizon, action_dim)")

# Clean up hooks
for h in hooks:
    h.remove()

## Visualizing the U-Net Architecture

The classic U-Net has a **U-shape**: the encoder goes down (reducing resolution, increasing channels), and the decoder goes back up (increasing resolution, decreasing channels). Skip connections bridge across the U.

Let's draw this for our 1D action U-Net.

In [ ]:
# ============================================================
# U-shaped architecture diagram
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 7)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('1D U-Net Architecture for Action Diffusion', fontsize=16, fontweight='bold', pad=20)

# Colors
encoder_color = '#5B8CB8'   # blue
decoder_color = '#D97757'   # accent
bottleneck_color = '#9B7EC8'  # purple
skip_color = '#7DA488'      # teal
io_color = '#C4956A'        # warm

# Define block positions (x, y, width, height, label, channels, temporal)
# Encoder blocks go down-left, decoder blocks go up-right
blocks = [
    # Encoder
    (0.5, 5.5, 1.8, 0.8, 'Input\nConv1d', '2 -> 256', 'T=16', encoder_color),
    (1.5, 4.0, 1.8, 0.8, 'Down Block 1\nResBlock + DS', '256 -> 512', 'T=8', encoder_color),
    (2.5, 2.5, 1.8, 0.8, 'Down Block 2\nResBlock + DS', '512 -> 1024', 'T=4', encoder_color),
    # Bottleneck
    (4.2, 1.2, 1.8, 0.8, 'Mid Block\nResBlock x2', '1024', 'T=4', bottleneck_color),
    # Decoder
    (5.9, 2.5, 1.8, 0.8, 'Up Block 1\nResBlock + US', '1024 -> 512', 'T=8', decoder_color),
    (6.9, 4.0, 1.8, 0.8, 'Up Block 2\nResBlock + US', '512 -> 256', 'T=16', decoder_color),
    (7.9, 5.5, 1.8, 0.8, 'Final\nConv1d', '256 -> 2', 'T=16', decoder_color),
]

# Draw blocks
for x, y, w, h, label, channels, temporal, color in blocks:
    rect = plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='white',
                         facecolor=color, alpha=0.85, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + 0.05, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white', zorder=3)
    ax.text(x + w/2, y - 0.15, f'{channels}', ha='center', va='top',
            fontsize=7, color=color, fontstyle='italic')
    ax.text(x + w/2, y - 0.35, f'{temporal}', ha='center', va='top',
            fontsize=7, color='gray')

# Draw arrows (down path)
arrow_props = dict(arrowstyle='->', color='white', lw=2)
ax.annotate('', xy=(1.8, 4.8), xytext=(1.5, 5.5), arrowprops=arrow_props)
ax.annotate('', xy=(2.8, 3.3), xytext=(2.5, 4.0), arrowprops=arrow_props)
ax.annotate('', xy=(4.5, 2.0), xytext=(3.8, 2.5), arrowprops=arrow_props)

# Draw arrows (up path)
ax.annotate('', xy=(6.2, 2.5), xytext=(5.7, 2.0), arrowprops=arrow_props)
ax.annotate('', xy=(7.5, 4.0), xytext=(7.2, 3.3), arrowprops=arrow_props)
ax.annotate('', xy=(8.5, 5.5), xytext=(8.2, 4.8), arrowprops=arrow_props)

# Draw skip connections
skip_props = dict(arrowstyle='->', color=skip_color, lw=2, linestyle='dashed')
# Skip from down block 0 to up block 2
ax.annotate('', xy=(7.9, 5.9), xytext=(2.3, 5.9),
            arrowprops=dict(arrowstyle='->', color=skip_color, lw=2, linestyle='dashed',
                           connectionstyle='arc3,rad=0.15'))
ax.text(5.1, 6.4, 'Skip Connection (concat)', ha='center', fontsize=8, color=skip_color)

# Skip from down block 1 to up block 1
ax.annotate('', xy=(6.9, 4.4), xytext=(3.3, 4.4),
            arrowprops=dict(arrowstyle='->', color=skip_color, lw=2, linestyle='dashed',
                           connectionstyle='arc3,rad=0.15'))
ax.text(5.1, 4.9, 'Skip Connection (concat)', ha='center', fontsize=8, color=skip_color)

# Skip from down block 2 to up block 0
ax.annotate('', xy=(5.9, 2.9), xytext=(4.3, 2.9),
            arrowprops=dict(arrowstyle='->', color=skip_color, lw=2, linestyle='dashed',
                           connectionstyle='arc3,rad=0.15'))
ax.text(5.1, 3.3, 'Skip (concat)', ha='center', fontsize=8, color=skip_color)

# Input/Output labels
ax.text(1.4, 6.6, 'Noisy Actions\n[B, 2, 16]', ha='center', fontsize=9,
        color=io_color, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#1A1915', edgecolor=io_color))
ax.text(8.8, 6.6, 'Predicted Noise\n[B, 2, 16]', ha='center', fontsize=9,
        color=io_color, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#1A1915', edgecolor=io_color))

# Conditioning arrow
ax.text(5.1, 0.3, 'Timestep + Obs Conditioning\n(via FiLM at every ResBlock)', ha='center',
        fontsize=9, color='#9B7EC8', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#1A1915', edgecolor='#9B7EC8'))

fig.patch.set_facecolor('#1A1915')
ax.set_facecolor('#1A1915')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Full forward pass: input noise vs predicted noise
# ============================================================

with torch.no_grad():
    # Create synthetic observation conditioning
    # (In the real policy, this comes from ResNet18 + SpatialSoftmax + state encoder)
    obs_cond = torch.randn(1, global_cond_dim).to(device)

    # Create random noise (this simulates a noisy action trajectory)
    input_noise = torch.randn(1, action_dim, horizon).to(device)
    timestep_val = torch.tensor([500]).to(device)  # Mid-way through diffusion

    # Forward pass through U-Net
    predicted_noise = unet(input_noise, timestep_val, local_cond=None, global_cond=obs_cond)

print(f"Input noise shape:     {list(input_noise.shape)}")
print(f"Predicted noise shape: {list(predicted_noise.shape)}")
print(f"Input noise range:     [{input_noise.min():.3f}, {input_noise.max():.3f}]")
print(f"Predicted noise range: [{predicted_noise.min():.3f}, {predicted_noise.max():.3f}]")

# Convert to numpy for plotting
input_np = input_noise.squeeze(0).cpu().numpy()       # [2, 16]
predicted_np = predicted_noise.squeeze(0).cpu().numpy()  # [2, 16]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#1A1915')
fig.suptitle('U-Net Forward Pass: Input Noise vs Predicted Noise', fontsize=14,
             fontweight='bold', color='white')

timesteps = np.arange(horizon)
dim_labels = ['X (dim 0)', 'Y (dim 1)']
colors_in = ['#5B8CB8', '#7DA488']
colors_pred = ['#D97757', '#9B7EC8']

for dim in range(2):
    # Left column: time series
    ax = axes[dim, 0]
    ax.set_facecolor('#2A2A25')
    ax.plot(timesteps, input_np[dim], 'o-', color=colors_in[dim], linewidth=2,
            markersize=5, label=f'Input Noise ({dim_labels[dim]})', alpha=0.8)
    ax.plot(timesteps, predicted_np[dim], 's--', color=colors_pred[dim], linewidth=2,
            markersize=5, label=f'Predicted Noise ({dim_labels[dim]})', alpha=0.8)
    ax.set_xlabel('Timestep', color='white')
    ax.set_ylabel('Value', color='white')
    ax.set_title(f'{dim_labels[dim]} - Time Series', color='white')
    ax.legend(fontsize=8)
    ax.tick_params(colors='white')
    ax.grid(True, alpha=0.2)
    for spine in ax.spines.values():
        spine.set_color('gray')

# Right column: scatter comparison
ax = axes[0, 1]
ax.set_facecolor('#2A2A25')
ax.scatter(input_np[0], predicted_np[0], c=colors_in[0], s=50, alpha=0.8, label='X dim')
ax.scatter(input_np[1], predicted_np[1], c=colors_in[1], s=50, alpha=0.8, label='Y dim')
lims = [min(input_np.min(), predicted_np.min()) - 0.5, max(input_np.max(), predicted_np.max()) + 0.5]
ax.plot(lims, lims, '--', color='gray', alpha=0.5, label='y=x (identical)')
ax.set_xlabel('Input Noise', color='white')
ax.set_ylabel('Predicted Noise', color='white')
ax.set_title('Input vs Predicted (scatter)', color='white')
ax.legend(fontsize=8)
ax.tick_params(colors='white')
ax.grid(True, alpha=0.2)
for spine in ax.spines.values():
    spine.set_color('gray')

# Difference plot
ax = axes[1, 1]
ax.set_facecolor('#2A2A25')
diff = predicted_np - input_np
ax.bar(timesteps - 0.15, diff[0], 0.3, color=colors_pred[0], alpha=0.8, label='X diff')
ax.bar(timesteps + 0.15, diff[1], 0.3, color=colors_pred[1], alpha=0.8, label='Y diff')
ax.axhline(y=0, color='white', linestyle='-', alpha=0.3)
ax.set_xlabel('Timestep', color='white')
ax.set_ylabel('Predicted - Input', color='white')
ax.set_title('Difference (how much U-Net changed the noise)', color='white')
ax.legend(fontsize=8)
ax.tick_params(colors='white')
ax.grid(True, alpha=0.2)
for spine in ax.spines.values():
    spine.set_color('gray')

plt.tight_layout()
plt.show()

print("\nKey insight: The U-Net output is NOT the same as the input noise!")
print("The U-Net has learned (or will learn during training) to predict the")
print("actual noise that was added, given the conditioning information.")
print(f"\nMean absolute difference: {np.abs(diff).mean():.4f}")
print("(With random weights, the output is essentially random -- after training,")
print("the U-Net would accurately predict the noise that was added.)")

In [ ]:
# ============================================================
# Bonus: Run the U-Net at different diffusion timesteps
# ============================================================
# The U-Net's behavior changes depending on the diffusion timestep.
# At high timesteps (very noisy), it makes large corrections.
# At low timesteps (nearly clean), it makes small refinements.

timesteps_to_test = [0, 100, 250, 500, 750, 999]
fixed_noise = torch.randn(1, action_dim, horizon).to(device)

# Use a fixed synthetic conditioning vector
obs_cond_fixed = torch.randn(1, global_cond_dim).to(device)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('#1A1915')
fig.suptitle('U-Net Output at Different Diffusion Timesteps', fontsize=14,
             fontweight='bold', color='white')

with torch.no_grad():
    for idx, t_val in enumerate(timesteps_to_test):
        t_tensor = torch.tensor([t_val]).to(device)
        pred = unet(fixed_noise, t_tensor, local_cond=None, global_cond=obs_cond_fixed)
        pred_np = pred.squeeze(0).cpu().numpy()

        ax = axes[idx // 3, idx % 3]
        ax.set_facecolor('#2A2A25')
        time_axis = np.arange(horizon)
        ax.plot(time_axis, pred_np[0], 'o-', color='#D97757', linewidth=2,
                markersize=4, label='X pred')
        ax.plot(time_axis, pred_np[1], 's-', color='#5B8CB8', linewidth=2,
                markersize=4, label='Y pred')
        ax.set_title(f't = {t_val}', color='white', fontsize=12)
        ax.set_ylim(-4, 4)
        ax.tick_params(colors='white')
        ax.grid(True, alpha=0.2)
        ax.legend(fontsize=7)
        for spine in ax.spines.values():
            spine.set_color('gray')

        # Print magnitude
        magnitude = np.abs(pred_np).mean()
        ax.text(0.95, 0.05, f'|output| = {magnitude:.3f}', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=8, color='#C4956A',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#1A1915', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nNotice how the U-Net output pattern changes with the diffusion timestep.")
print("The timestep embedding tells the U-Net how noisy the input is,")
print("so it can calibrate its noise prediction accordingly.")
print("\n(With random weights, the variation is due to the timestep embedding")
print("affecting the FiLM conditioning differently at each timestep.)")

## Key Takeaways

1. **1D, not 2D**: Action trajectories are 1D sequences, so we use `Conv1d` instead of `Conv2d`
2. **Encoder-Decoder with Skip Connections**: The U-shape compresses then reconstructs, with skip connections preserving detail
3. **Conditioning is everywhere**: Timestep and observation embeddings are injected at every residual block (via FiLM -- see the next notebook!)
4. **Same input/output shape**: Input `[B, 2, 16]` -> Output `[B, 2, 16]` -- the U-Net predicts noise with the same shape as the noisy actions

## TODO Exercise

Try changing the number of channels in `down_dims` (the default is `(256, 512, 1024)`). 

```python
# Try a smaller U-Net:
unet_small = ConditionalUnet1D(
    input_dim=2, global_cond_dim=260,
    down_dims=(64, 128, 256),
).to(device)
print(f"Small U-Net params: {sum(p.numel() for p in unet_small.parameters()):,}")

# Compare with the default:
print(f"Default U-Net params: {sum(p.numel() for p in unet.parameters()):,}")
```

- What happens with `(64, 128, 256)`? How many parameters does that give?
- What is the **minimum** channel configuration that still trains successfully on PushT?